# Value-for-Money QLoRA — Fine-Tuning Notebook (Kaggle T4 x2)

Fine-tunes an open-source model to predict a wine's value-for-money (VFM)
score, 0-99, from its tasting note and metadata.

**Hardcoded for free Kaggle GPU (T4 x2 recommended — set the Accelerator in the right sidebar before running).** The full curated dataset is small
enough that the heavier LoRA config (rank 256, attention + MLP layers)
fits — the T4 constraint is memory, handled here via a small
per-device batch size plus gradient accumulation, and by keeping the
optimizer math in fp32 (which avoids the fp16 gradient scaler entirely).

**Setup:** clone your repo in the first cell below (needs `utils/` on the path). Add `HF_TOKEN` (and `GITHUB_TOKEN` if the repo is private) under Add-ons > Secrets in the Kaggle notebook editor. Set Accelerator to `GPU T4 x2` in the session options before running.

**Run mode:** for a run longer than a live session, use **Save Version > Save & Run All (Commit)** instead of running cells interactively — this keeps training going on Kaggle's servers even if you close the browser.

| Part | Content |
|---|---|
| 1 | Exploration: quantization memory footprint, LoRA parameter counting |
| 2 | Tokenizer sanity check |
| 3 | Training: QLoRA fine-tune via SFTTrainer |
| 4 | Testing: load the trained adapter, evaluate on the held-out test set |

### Why fp32 optimizer math (the crash fix)

The fp16 mixed-precision path relies on a gradient scaler that is not
implemented for some dtypes on a T4 (Turing, compute capability 7.5),
producing a `NotImplementedError` in `_amp_foreach_non_finite_check...`.
Setting `fp16=False, bf16=False` keeps the base weights 4-bit quantized
(so memory stays low) while the small trainable LoRA adapters use fp32 —
no scaler is ever invoked, so that error cannot occur.


## Part 1 — Exploration: Quantization and LoRA sizing

Two things worth building intuition for before training: how much memory
quantization saves, and how many trainable parameters LoRA actually adds.

**Note:** loading the base model at multiple precisions in one session
can exhaust GPU memory. If you hit a CUDA OOM here, use Runtime > Restart
session between the three loads. These exploration cells are illustrative
and do not affect training — Part 3 reloads the model from scratch.


In [ ]:
!pip install -q --upgrade bitsandbytes trl peft

from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()

# If your repo is private, set a GITHUB_TOKEN secret; if public, this
# works without it — just drop the token from the URL.
github_token = secrets.get_secret("GITHUB_TOKEN")
!git clone https://{github_token}@github.com/gtraskas/YOUR_REPO.git
%cd YOUR_REPO


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
hf_token = secrets.get_secret('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

BASE_MODEL: str = "meta-llama/Llama-3.2-3B"


In [ ]:
# ── Load at full precision — establishes the memory baseline ──────────────────
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, device_map="auto")
print(f"Full precision memory footprint: {base_model.get_memory_footprint() / 1e9:,.2f} GB")


In [ ]:
# ── Restart session (Runtime > Restart session), then load at 8-bit ──────────
quant_config_8bit = BitsAndBytesConfig(load_in_8bit=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config_8bit, device_map="auto",
)
print(f"8-bit memory footprint: {base_model.get_memory_footprint() / 1e9:,.2f} GB")


In [ ]:
# ── Restart session again, then load at 4-bit — this is what training uses ───
# Compute dtype is float16 (T4 does not support bfloat16).
quant_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, quantization_config=quant_config_4bit, device_map="auto",
)
print(f"4-bit memory footprint: {base_model.get_memory_footprint() / 1e9:,.2f} GB")


In [ ]:
# ── LoRA parameter counting — how much does the adapter actually add? ─────────
# Llama 3.2 3B dims: hidden=3072, q/o project to 3072, k/v to 1024
# (grouped-query attention), MLP to 8192, across 28 transformer layers.

def count_lora_params(r: int, include_mlp: bool) -> tuple[int, float]:
    """Count LoRA adapter parameters for a given rank and layer scope."""
    lora_q_proj = 3072 * r + 3072 * r
    lora_k_proj = 3072 * r + 1024 * r
    lora_v_proj = 3072 * r + 1024 * r
    lora_o_proj = 3072 * r + 3072 * r
    layer_params = lora_q_proj + lora_k_proj + lora_v_proj + lora_o_proj

    if include_mlp:
        lora_gate_proj = 3072 * r + 8192 * r
        lora_up_proj = 3072 * r + 8192 * r
        lora_down_proj = 3072 * r + 8192 * r
        layer_params += lora_gate_proj + lora_up_proj + lora_down_proj

    total_params = layer_params * 28
    size_mb = (total_params * 4) / 1_000_000
    return total_params, size_mb

params, size_mb = count_lora_params(r=256, include_mlp=True)
print(f"Attention + MLP, r=256: {params:,} params, {size_mb:,.1f} MB")
# A tiny fraction of the ~3 BILLION base params — that is why it fits.


## Part 2 — Tokenizer Sanity Check

Wine summaries are a bounded, templated format. This confirms the token
cutoff needs little or no truncation before committing to it.


In [ ]:
import matplotlib.pyplot as plt
from utils.items import Wine
from utils.vfm import apply_vfm

HUB_DATASET_NAME: str = "gtraskas/wine-vfm-curated"
train, val, test = Wine.from_hub(HUB_DATASET_NAME)
for split in (train, val, test):
    apply_vfm(split)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

token_counts = [w.count_tokens(tokenizer) for w in train]
plt.figure(figsize=(15, 5))
plt.title(f"Summary tokens: avg {sum(token_counts)/len(token_counts):.1f}, max {max(token_counts)}")
plt.hist(token_counts, bins=40, color="#4A6FA5")
plt.xlabel("Tokens"); plt.ylabel("Count")
plt.show()


In [ ]:
CUTOFF: int = 256  # generous; the histogram above should show near-zero truncation
over_cutoff = sum(1 for c in token_counts if c > CUTOFF)
print(f"{over_cutoff} of {len(token_counts)} summaries exceed {CUTOFF} tokens "
      f"({over_cutoff / len(token_counts):.1%})")


## Part 3 — Training

Build `prompt`/`completion` pairs, quantize the base model to 4-bit,
attach LoRA adapters (attention + MLP, rank 256), and run `SFTTrainer`
with fp32 optimizer math.


In [ ]:
for split in (train, val, test):
    for wine in split:
        wine.make_training_prompt(tokenizer, CUTOFF, do_round=True)

print(train[0].prompt)
print("---")
print(train[0].completion)


In [ ]:
import os
from datetime import datetime

from datasets import Dataset
from peft import LoraConfig
from transformers import set_seed
from trl import SFTConfig, SFTTrainer

# ── Constants (hardcoded for T4) ───────────────────────────────────────────────
PROJECT_NAME: str = "wine-vfm"
HF_USER: str = "gtraskas"

RUN_NAME: str = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
PROJECT_RUN_NAME: str = f"{PROJECT_NAME}-{RUN_NAME}"
HUB_MODEL_NAME: str = f"{HF_USER}/{PROJECT_RUN_NAME}"

# Overall hyperparameters
EPOCHS: int = 1
BATCH_SIZE: int = 4                     # small per-device batch to fit T4 memory
GRADIENT_ACCUMULATION_STEPS: int = 8    # effective batch = 4 * 8 = 32
MAX_SEQUENCE_LENGTH: int = CUTOFF + 20  # margin for the prompt scaffolding

# QLoRA hyperparameters — full adapter (dataset is small, this fits)
LORA_R: int = 256
LORA_ALPHA: int = LORA_R * 2
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj",
                  "gate_proj", "up_proj", "down_proj"]
LORA_DROPOUT: float = 0.1

# Training hyperparameters
LEARNING_RATE: float = 1e-4
WARMUP_RATIO: float = 0.01
LR_SCHEDULER_TYPE: str = "cosine"
WEIGHT_DECAY: float = 0.001
OPTIMIZER: str = "paged_adamw_32bit"

# Tracking
VAL_SIZE: int = 500
LOG_STEPS: int = 10
SAVE_STEPS: int = 200


In [ ]:
# ── Build the HF Dataset objects the trainer expects ───────────────────────────
train_ds = Dataset.from_list([w.to_datapoint() for w in train])
val_ds = Dataset.from_list([w.to_datapoint() for w in val[:VAL_SIZE]])
test_ds = Dataset.from_list([w.to_datapoint() for w in test])


In [ ]:
# ── Quantized base model — 4-bit, float16 compute (T4-safe) ───────────────────
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
)

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
    torch_dtype=torch.float32,  # non-quantized weights (incl. LoRA) in fp32
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
print(f"Memory footprint: {base_model.get_memory_footprint() / 1e6:,.1f} MB")


In [ ]:
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    fp16=False,   # fp32 optimizer math — no gradient scaler, no T4 dtype crash
    bf16=False,   # T4 does not support bfloat16
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type=LR_SCHEDULER_TYPE,
    report_to="none",
    run_name=RUN_NAME,
    max_length=MAX_SEQUENCE_LENGTH,
    save_strategy="steps",
    hub_strategy="every_save",
    push_to_hub=True,
    hub_model_id=HUB_MODEL_NAME,
    hub_private_repo=True,
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
)

fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    peft_config=lora_parameters,
    args=train_parameters,
)


In [ ]:
# ── Train — pushes a checkpoint to the Hub every SAVE_STEPS, so a Colab ───────
# disconnect doesn't lose progress. Resume by reloading HUB_MODEL_NAME.
set_seed(42)
fine_tuning.train()

fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the Hub: {PROJECT_RUN_NAME}")


## Part 4 — Testing

Load the frozen 4-bit base model plus the trained LoRA adapter, generate
a VFM prediction per test wine, and run it through the shared `Tester`/
`evaluate` harness — MAE ± 95% CI, R², scatter, error trend.


In [ ]:
from peft import PeftModel

from utils.evaluator import Tester, evaluate

REVISION: str | None = None  # set to a commit hash to pin a specific checkpoint

fine_tuned_model = (
    PeftModel.from_pretrained(base_model, HUB_MODEL_NAME, revision=REVISION)
    if REVISION else
    PeftModel.from_pretrained(base_model, HUB_MODEL_NAME)
)
print(f"Memory footprint: {fine_tuned_model.get_memory_footprint() / 1e6:,.1f} MB")


In [ ]:
from utils.items import PREFIX, QUESTION

def model_predict(wine) -> str:
    """Generate a VFM prediction from the fine-tuned adapter.

    Accepts a Wine object (matches the shared evaluator interface) and
    builds the inference prompt fresh, so there is no dependency on
    train-time prompt/completion state.
    """
    prompt = f"{QUESTION}\n\n{wine.summary}\n\n{PREFIX}"
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        output_ids = fine_tuned_model.generate(**inputs, max_new_tokens=8)
    prompt_len = inputs["input_ids"].shape[1]
    generated_ids = output_ids[0, prompt_len:]
    return tokenizer.decode(generated_ids)


In [ ]:
set_seed(42)
evaluate(model_predict, test)


## Results Log

Copy this rung's MAE/CI/R² into the master Results Log alongside the
other models for the final comparison chart.
